## مختبر ذاكرة الوكيل

في هذا المختبر سنبني تدريجياً مكونات ذاكرة وكيل محادثة. سنغطي تخزين المحادثات واسترجاعها، والمحاكاة البسيطة للذاكرة المعتمدة على تشابه الكلمات، وإدارة نافذة السياق، ودمج كل ذلك لبناء موجه وكيل متكامل.

## الجزء الأول: بناء مخزن محادثة بسيط

سنقوم بتنفيذ صنف `ConversationMemory` الذي يدعم إضافة الرسائل واسترجاع آخر `k` رسالة.

In [ ]:
from typing import List, Dict

class ConversationMemory:
    def __init__(self, max_size: int = 100):
        self.messages: List[Dict[str, str]] = []
        self.max_size = max_size
    
    def add_message(self, role: str, content: str):
        # TODO: أضف الرسالة إلى القائمة، وإذا تجاوزت max_size احذف أقدم رسالة
        pass
    
    def get_last_k(self, k: int) -> List[Dict[str, str]]:
        # TODO: أرجع آخر k رسالة
        pass

In [ ]:
# اختبار ConversationMemory بعد إكماله
memory = ConversationMemory(max_size=5)
memory.add_message("user", "مرحباً")
memory.add_message("assistant", "أهلاً بك! كيف يمكنني مساعدتك؟")
memory.add_message("user", "ما هو الطقس اليوم؟")
memory.add_message("assistant", "الطقس مشمس.")
memory.add_message("user", "شكراً")
memory.add_message("assistant", "عفواً")

# استرجاع آخر 3 رسائل
last_three = memory.get_last_k(3)
for msg in last_three:
    print(f"{msg['role']}: {msg['content']}")

## الجزء الثاني: محاكاة ذاكرة متجهات باستخدام الكلمات المفتاحية

سنقوم ببناء دالة تحسب التشابه بين استعلام ومستند بناءً على تداخل الكلمات (معامل جاكارد)، ودالة لاسترجاع أكثر المستندات صلة.

In [ ]:
# مجموعة من المستندات النصية
documents = [
    "The cat sat on the mat.",
    "Dogs are loyal animals.",
    "Cars are fast vehicles.",
    "The quick brown fox jumps over the lazy dog.",
    "Python programming is fun.",
    "Machine learning is a subset of artificial intelligence.",
    "The sun is shining bright today.",
]

In [ ]:
def keyword_overlap_similarity(query: str, document: str) -> float:
    # TODO: حول الاستعلام والمستند إلى مجموعات كلمات (lowercase, split) واحسب معامل جاكارد (Jaccard)
    pass

def retrieve_top_k(query: str, documents: List[str], k: int = 2) -> List[str]:
    # TODO: احسب التشابه لكل مستند وأعد أعلى k مستندات
    pass

In [ ]:
# اختبار استرجاع المستندات
query = "quick brown fox"
top_docs = retrieve_top_k(query, documents, k=2)
print("Top matching documents:")
for i, doc in enumerate(top_docs, 1):
    print(f"{i}: {doc}")

## الجزء الثالث: إدارة نافذة السياق

سنطبق دالة تقوم باقتطاع قائمة الرسائل بحيث لا يتجاوز العدد الإجمالي للكلمات حداً معيناً، مع الاحتفاظ بأحدث الرسائل.

In [ ]:
def truncate_messages(messages: List[Dict[str, str]], max_tokens: int) -> List[Dict[str, str]]:
    # TODO: احذف أقدم الرسائل بحيث لا يتجاوز مجموع عدد الكلمات max_tokens
    # افترض أن كل رسالة تُحسب بعدد كلمات content (role لا يُحتسب)
    pass

In [ ]:
# اختبار الاقتطاع
long_conversation = [
    {"role": "user", "content": "Hello"},
    {"role": "assistant", "content": "Hi there!"},
    {"role": "user", "content": "Tell me a long story about dragons."},
    {"role": "assistant", "content": "Once upon a time, there was a dragon who lived in a mountain. The dragon loved to fly over the village and greet the children. They all adored the dragon because it brought them gifts from distant lands."},
    {"role": "user", "content": "That's great!"},
]

truncated = truncate_messages(long_conversation, max_tokens=15)
print("Truncated conversation (max tokens=15):")
for msg in truncated:
    print(f"{msg['role']}: {msg['content']}")

## الجزء الرابع: دمج الذاكرة في وكيل بسيط

سنقوم بإنشاء وظيفة `build_prompt` التي تجمع ذاكرة المحادثة وذاكرة المستندات لتكوين موجه متكامل يمكن إرساله لنموذج لغوي.

In [ ]:
class DocumentMemory:
    def __init__(self, documents: List[str]):
        self.documents = documents
    
    def retrieve(self, query: str, k: int = 2) -> List[str]:
        return retrieve_top_k(query, self.documents, k)


def build_prompt(user_query: str, conv_memory: ConversationMemory, doc_memory: DocumentMemory, max_history_k: int = 3, max_docs: int = 2) -> str:
    # TODO: قم ببناء النص التالي الذي سيُرسل للنموذج اللغوي:
    # - يمكن تضمين مقدمة نظام (system message) تشرح المهمة
    # - استرجاع المستندات ذات الصلة من doc_memory بناءً على user_query
    # - استخراج آخر max_history_k رسالة من conv_memory
    # - تجميع prompt مرتب
    pass

In [ ]:
# اختبار build_prompt
conv_mem = ConversationMemory(max_size=10)
conv_mem.add_message("user", "What is a cat?")
conv_mem.add_message("assistant", "A cat is a small domesticated carnivorous mammal.")
conv_mem.add_message("user", "Tell me more about dogs.")
conv_mem.add_message("assistant", "Dogs are loyal and friendly animals.")

doc_mem = DocumentMemory(documents)

user_input = "Tell me about quick brown fox"
prompt = build_prompt(user_input, conv_mem, doc_mem, max_history_k=2, max_docs=2)
print(prompt)